In [ ]:
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path

In [40]:

con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")

files = sorted(Path("data").glob("*.parquet"))

for f in files:
    cols = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(f)]
    ).fetchall()
    print(f"{f.name}: {len(cols)} колонок")

print("\nВсе колонки первого файла:")
if files:
    print([c[0] for c in con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(files[0])]
    ).fetchall()])


год=2019.parquet: 223 колонок
год=2020.parquet: 223 колонок
год=2021.parquet: 223 колонок
год=2022.parquet: 223 колонок
год=2023.parquet: 223 колонок
год=2024.parquet: 223 колонок

Все колонки первого файла:
['inn', 'eligible', 'exemption_criteria', 'filed', 'financial', 'simplified', 'line_1100', 'line_1105', 'line_1110', 'line_1120', 'line_1130', 'line_1140', 'line_1150', 'line_1160', 'line_1170', 'line_1180', 'line_1190', 'line_1200', 'line_1210', 'line_1215', 'line_1220', 'line_1230', 'line_1240', 'line_1250', 'line_1260', 'line_1300', 'line_1310', 'line_1320', 'line_1330', 'line_1340', 'line_1350', 'line_1360', 'line_1370', 'line_1400', 'line_1410', 'line_1420', 'line_1430', 'line_1450', 'line_1500', 'line_1510', 'line_1520', 'line_1530', 'line_1540', 'line_1550', 'line_1600', 'line_1700', 'line_2100', 'line_2110', 'line_2120', 'line_2200', 'line_2210', 'line_2220', 'line_2300', 'line_2310', 'line_2320', 'line_2330', 'line_2340', 'line_2350', 'line_2400', 'line_2410', 'line_2411',

In [41]:
con = duckdb.connect()
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")

p = "data/*.parquet"

print(con.execute(f"""
SELECT
    filed,
    imputed,
    exemption_criteria,
    COUNT(*) AS n
FROM read_parquet('{p}', union_by_name=true)
GROUP BY 1, 2, 3
ORDER BY n DESC
""").df())

    filed  imputed exemption_criteria         n
0     1.0      0.0               none  12600240
1     0.0      0.0               none   7616442
2     0.0      1.0               none    397282
3     0.0      0.0          initiated    158056
4     1.0      0.0          initiated    102030
5     0.0      0.0              state     32615
6     1.0      0.0              state     21723
7     0.0      1.0          initiated     15689
8     1.0      0.0          financial      2301
9     0.0      1.0              state      2028
10    0.0      0.0          financial       339
11    0.0      1.0          financial       307
12    1.0      0.0          religious       176
13    0.0      0.0          religious        73
14    0.0      1.0          religious        23


In [42]:
print(con.execute("""
SELECT
    LEFT(CAST(okved AS VARCHAR), 2) AS sector,
    COUNT(*) AS n
FROM read_parquet('data/*.parquet', union_by_name=true)
WHERE filed = 1
  AND exemption_criteria = 'none'
  AND LEFT(CAST(okved AS VARCHAR), 2)
      IN ('64', '65', '66')
GROUP BY 1
ORDER BY 1
""").df())

  sector       n
0     64  107913
1     66   34384


In [43]:
con.execute("""
CREATE OR REPLACE VIEW panel AS
SELECT *
FROM read_parquet('data/*.parquet', union_by_name=true)
WHERE filed = 1
  AND exemption_criteria NOT IN ('financial', 'state', 'religious')
  AND (
      okved IS NULL
      OR LEFT(CAST(okved AS VARCHAR), 2)
         NOT IN ('64', '65', '66')
  )
""")

In [44]:
con.execute("""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT inn) AS companies,
    COUNT(*) FILTER (WHERE okved IS NULL) AS missing_okved
FROM panel
""").df()

,observations,companies,missing_okved
0,12558775,3137552,78


In [45]:
cols = con.execute("DESCRIBE SELECT * FROM panel").df()
print(cols["column_name"].tolist())

['inn', 'eligible', 'exemption_criteria', 'filed', 'financial', 'simplified', 'line_1100', 'line_1105', 'line_1110', 'line_1120', 'line_1130', 'line_1140', 'line_1150', 'line_1160', 'line_1170', 'line_1180', 'line_1190', 'line_1200', 'line_1210', 'line_1215', 'line_1220', 'line_1230', 'line_1240', 'line_1250', 'line_1260', 'line_1300', 'line_1310', 'line_1320', 'line_1330', 'line_1340', 'line_1350', 'line_1360', 'line_1370', 'line_1400', 'line_1410', 'line_1420', 'line_1430', 'line_1450', 'line_1500', 'line_1510', 'line_1520', 'line_1530', 'line_1540', 'line_1550', 'line_1600', 'line_1700', 'line_2100', 'line_2110', 'line_2120', 'line_2200', 'line_2210', 'line_2220', 'line_2300', 'line_2310', 'line_2320', 'line_2330', 'line_2340', 'line_2350', 'line_2400', 'line_2410', 'line_2411', 'line_2412', 'line_2420', 'line_2421', 'line_2430', 'line_2450', 'line_2460', 'line_2500', 'line_2510', 'line_2520', 'line_2530', 'line_2900', 'line_2910', 'line_3100', 'line_3101', 'line_3110', 'line_3120',

In [46]:
con.execute("""
CREATE OR REPLACE VIEW panel AS
SELECT *
FROM read_parquet('data/*.parquet', union_by_name=true, filename=true)
WHERE filed = 1
  AND exemption_criteria NOT IN ('financial', 'state', 'religious')
  AND (okved IS NULL OR 
         LEFT(CAST(okved AS VARCHAR), 2) NOT IN ('64', '65', '66')
  )
""")

In [47]:
con.execute("""
SELECT
    filename,
    COUNT(*) AS observations,
    COUNT(DISTINCT inn) AS companies
FROM panel
GROUP BY filename
ORDER BY filename
""").df()

,filename,observations,companies
0,data/год=2019.parquet,2032347,2032347
1,data/год=2020.parquet,2167857,2167857
2,data/год=2021.parquet,2123044,2123044
3,data/год=2022.parquet,2077443,2077443
4,data/год=2023.parquet,2082021,2082021
5,data/год=2024.parquet,2076063,2076063


In [48]:
con.execute("""
CREATE OR REPLACE VIEW panel_year AS
SELECT *,
    CAST(
        regexp_extract(filename, 'год=([0-9]{4})', 1)
        AS INTEGER
    ) AS year
FROM panel
""")

In [49]:
con.execute("""
SELECT
    year,
    COUNT(*) AS observations,
    COUNT(DISTINCT inn) AS companies
FROM panel_year
GROUP BY year
ORDER BY year
""").df()

,year,observations,companies
0,2019,2032347,2032347
1,2020,2167857,2167857
2,2021,2123044,2123044
3,2022,2077443,2077443
4,2023,2082021,2082021
5,2024,2076063,2076063


In [50]:
needed = [
    # Идентификаторы
    'inn', 'year',

    # Задолженность
    'line_1410', 'line_1510', 'line_1600',

    # Факторы модели
    'line_2110', 'line_2400', 'line_1150',
    'line_2411', 'line_2410',
    'age', 'okved', 'region_taxcode',

    # Дополнительно для EBIT
    'line_2300', 'line_2330',

    # Контроль качества
    'filed', 'imputed', 'simplified',
    'articulated', 'outlier',
    'totals_adjustment', 'exemption_criteria'
]

available = set(con.execute(
    "DESCRIBE SELECT * FROM panel_year"
).df()['column_name'])

print("Отсутствуют:", sorted(set(needed) - available))

Отсутствуют: []


In [51]:
cols = [c for c in needed if c in available]

con.execute(f"""
CREATE OR REPLACE VIEW panel_model AS
SELECT {', '.join(cols)}
FROM panel_year
""")

print("Сохранено столбцов:", len(cols))
print(cols)

Сохранено столбцов: 22
['inn', 'year', 'line_1410', 'line_1510', 'line_1600', 'line_2110', 'line_2400', 'line_1150', 'line_2411', 'line_2410', 'age', 'okved', 'region_taxcode', 'line_2300', 'line_2330', 'filed', 'imputed', 'simplified', 'articulated', 'outlier', 'totals_adjustment', 'exemption_criteria']


In [52]:
required = [
    # Идентификаторы
    'inn', 'year',

    # Leverage
    'line_1410', 'line_1510', 'line_1600',

    # Profitability (EBIT / Assets)
    'line_2300', 'line_2330',

    # Другие факторы
    'line_2110', 'line_1150', 'line_2411',
    'age', 'okved'
]

extra = [
    # Дополнительные показатели
    'line_2410', 'region_taxcode',

    # Качество данных
    'filed', 'imputed', 'simplified',
    'articulated', 'outlier',
    'totals_adjustment', 'exemption_criteria'
]

available = set(con.execute(
    "DESCRIBE SELECT * FROM panel_year"
).df()['column_name'])

print("Отсутствуют обязательные:", set(required) - available)
print("Отсутствуют дополнительные:", set(extra) - available)

Отсутствуют обязательные: set()
Отсутствуют дополнительные: set()


In [53]:
missing = set(required) - available
if missing:
    raise ValueError(f"Нет столбцов: {missing}")

cols = required + [c for c in extra if c in available]

con.execute(f"""
CREATE OR REPLACE VIEW panel_model AS
SELECT {', '.join(cols)}
FROM panel_year
""")

print("Столбцов:", len(cols))

Столбцов: 21


In [54]:
con.execute("""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT inn) AS companies,
    COUNT(DISTINCT year) AS years
FROM panel_model
""").df()

,observations,companies,years
0,12558775,3137552,6


In [55]:
variables = [
    'line_1410', 'line_1510', 'line_1600',
    'line_2300', 'line_2330', 'line_2110',
    'line_1150', 'line_2411',
    'age', 'okved'
]

q = """
SELECT year, COUNT(*) AS n,
""" + ",\n".join(
    f"ROUND(100.0 * COUNT(*) FILTER "
    f"(WHERE {c} IS NULL) / COUNT(*), 2) AS {c}"
    for c in variables
) + """
FROM panel_model
GROUP BY year
ORDER BY year
"""

con.execute(q).df()

,year,n,line_1410,line_1510,line_1600,line_2300,line_2330,line_2110,line_1150,line_2411,age,okved
0,2019,2032347,70.93,56.67,0.0,13.54,83.14,18.86,56.60,75.14,0.0,0.0
1,2020,2167857,70.81,58.75,0.0,14.59,84.54,20.60,57.89,80.35,0.0,0.0
2,2021,2123044,68.30,59.38,0.0,14.32,84.32,20.79,56.28,81.13,0.0,0.0
3,2022,2077443,65.21,59.59,0.0,14.13,83.50,20.74,54.68,81.64,0.0,0.0
4,2023,2082021,62.89,61.75,0.0,13.50,81.12,20.43,53.82,81.64,0.0,0.0
5,2024,2076063,61.15,63.33,0.0,13.43,79.87,20.69,53.34,81.76,0.0,0.0


In [56]:
con.execute("""
SELECT
    year,
    COUNT(*) FILTER (
        WHERE line_1600 <= 0
    ) AS bad_assets,
    COUNT(*) FILTER (
        WHERE line_2110 <= 0
    ) AS bad_revenue,
    COUNT(*) FILTER (
        WHERE line_2330 > 0
    ) AS interest_positive,
    COUNT(*) FILTER (
        WHERE line_2330 < 0
    ) AS interest_negative,
    COUNT(*) FILTER (
        WHERE line_2330 = 0
    ) AS interest_zero
FROM panel_model
GROUP BY year
ORDER BY year
""").df()

,year,bad_assets,bad_revenue,interest_positive,interest_negative,interest_zero
0,2019,43849,136181,0,201739,140989
1,2020,50028,150934,0,204781,130302
2,2021,48779,135392,0,211187,121795
3,2022,46577,128769,0,223127,119557
4,2023,44665,135536,0,238989,154058
5,2024,43106,132124,0,254547,163329


In [57]:
con.execute("""
SELECT
    year,
    simplified,
    COUNT(*) AS n,

    ROUND(100.0 * COUNT(*) FILTER (
        WHERE line_2300 IS NULL
    ) / COUNT(*), 1) AS missing_profit,

    ROUND(100.0 * COUNT(*) FILTER (
        WHERE line_2330 IS NULL
    ) / COUNT(*), 1) AS missing_interest,

    ROUND(100.0 * COUNT(*) FILTER (
        WHERE line_1410 IS NULL
          AND line_1510 IS NULL
    ) / COUNT(*), 1) AS missing_both_debts,

    ROUND(100.0 * COUNT(*) FILTER (
        WHERE line_1150 IS NULL
    ) / COUNT(*), 1) AS missing_fixed_assets,

    ROUND(100.0 * COUNT(*) FILTER (
        WHERE line_2411 IS NULL
    ) / COUNT(*), 1) AS missing_tax

FROM panel_model
GROUP BY year, simplified
ORDER BY year, simplified
""").df()

,year,simplified,n,missing_profit,missing_interest,missing_both_debts,missing_fixed_assets,missing_tax
0,2019,0.0,964083,16.6,83.2,49.5,55.9,47.6
1,2019,1.0,1068264,10.8,83.1,42.4,57.2,100.0
2,2020,0.0,968597,17.9,83.7,49.5,56.2,56.0
3,2020,1.0,1199260,11.9,85.2,43.9,59.3,100.0
4,2021,0.0,905828,17.6,82.9,48.3,54.3,55.8
5,2021,1.0,1217216,11.9,85.4,43.1,57.8,100.0
6,2022,0.0,852941,17.6,81.3,46.8,52.0,55.3
7,2022,1.0,1224502,11.7,85.1,42.2,56.5,100.0
8,2023,0.0,797079,14.5,77.2,45.8,49.5,52.0
9,2023,1.0,1284942,12.9,83.5,43.3,56.5,100.0


In [58]:
con.execute("""
SELECT
    year,
    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND line_1410 IS NULL
          AND line_1510 IS NULL
    ) AS no_interest_no_debt,

    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND (line_1410 IS NOT NULL
               OR line_1510 IS NOT NULL)
    ) AS no_interest_with_debt,

    COUNT(*) FILTER (
        WHERE line_2330 IS NOT NULL
          AND line_1410 IS NULL
          AND line_1510 IS NULL
    ) AS interest_no_debt

FROM panel_model
GROUP BY year
ORDER BY year
""").df()

,year,no_interest_no_debt,no_interest_with_debt,interest_no_debt
0,2019,893023,796596,36709
1,2020,968664,864110,37645
2,2021,927472,862590,34986
3,2022,877282,857477,38738
4,2023,879329,809645,42677
5,2024,865393,792794,49348


In [59]:
con.execute("""
SELECT
    year,
    simplified,
    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND (line_1410 > 0 OR line_1510 > 0)
    ) AS positive_debt_no_interest,

    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND (line_1410 = 0 OR line_1510 = 0)
          AND COALESCE(line_1410, 0) = 0
          AND COALESCE(line_1510, 0) = 0
    ) AS zero_reported_debt,

    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND line_1410 IS NULL
          AND line_1510 IS NULL
    ) AS both_debts_missing

FROM panel_model
GROUP BY year, simplified
ORDER BY year, simplified
""").df()

,year,simplified,positive_debt_no_interest,zero_reported_debt,both_debts_missing
0,2019,0.0,260929,72115,466608
1,2019,1.0,346961,110028,426415
2,2020,0.0,270909,67627,470268
3,2020,1.0,419097,100449,498396
4,2021,0.0,245984,73886,428942
5,2021,1.0,423770,113078,498530
6,2022,0.0,225228,80035,386223
7,2022,1.0,425940,120534,491059
8,2023,0.0,205114,58636,350347
9,2023,1.0,436310,104387,528982


In [60]:
con.execute("""
SELECT year,
    COUNT(*) AS total,

    COUNT(*) FILTER (
        WHERE line_1600 > 0
          AND line_2300 IS NOT NULL
          AND line_2330 IS NOT NULL
    ) AS valid_ebit,

    COUNT(*) FILTER (
        WHERE line_1600 > 0
          AND line_1410 IS NOT NULL
          AND line_1510 IS NOT NULL
    ) AS valid_debt,

    COUNT(*) FILTER (
        WHERE line_1600 > 0
          AND line_2300 IS NOT NULL
          AND line_2330 IS NOT NULL
          AND line_1410 IS NOT NULL
          AND line_1510 IS NOT NULL
          AND line_2110 > 0
          AND line_1150 IS NOT NULL
    ) AS full_core

FROM panel_model
GROUP BY year
ORDER BY year
""").df()

,year,total,valid_ebit,valid_debt,full_core
0,2019,2032347,339359,365044,136691
1,2020,2167857,331135,362001,126717
2,2021,2123044,329164,371052,129450
3,2022,2077443,339157,396357,134755
4,2023,2082021,388174,404774,162646
5,2024,2076063,413082,402280,167488


In [61]:
con.execute("""
SELECT year,
    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND line_4123 IS NOT NULL
    ) AS missing_2330_has_4123,

    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND line_4224 IS NOT NULL
    ) AS missing_2330_has_4224,

    COUNT(*) FILTER (
        WHERE line_2330 IS NULL
          AND (line_4123 < 0 OR line_4224 < 0)
    ) AS missing_2330_paid_interest

FROM panel_year
GROUP BY year
ORDER BY year
""").df()

,year,missing_2330_has_4123,missing_2330_has_4224,missing_2330_paid_interest
0,2019,11195,2438,6476
1,2020,12331,3668,6173
2,2021,13135,4121,6519
3,2022,18444,9238,6719
4,2023,11371,2630,6552
5,2024,10597,2453,6286


In [62]:
con.execute("""
CREATE OR REPLACE VIEW panel_clean AS
SELECT *
FROM panel_model
WHERE outlier IS DISTINCT FROM 1
""")

In [63]:
con.execute("""
SELECT
    year,
    simplified,
    articulated,
    COUNT(*) AS n,
    COUNT(*) FILTER (
        WHERE totals_adjustment = 1
    ) AS adjusted
FROM panel_clean
GROUP BY year, simplified, articulated
ORDER BY year, simplified, articulated
""").df()

,year,simplified,articulated,n,adjusted
0,2019,0.0,0.0,32051,4328
1,2019,0.0,1.0,931989,376973
2,2019,1.0,0.0,45101,43944
3,2019,1.0,1.0,1023132,1020420
4,2020,0.0,0.0,34304,2893
5,2020,0.0,1.0,934262,185761
6,2020,1.0,0.0,63191,61606
7,2020,1.0,1.0,1136052,1131790
8,2021,0.0,0.0,31899,2164
9,2021,0.0,1.0,873910,163182


In [64]:
con.execute("""
CREATE OR REPLACE VIEW panel_quality AS
SELECT *
FROM panel_model
WHERE outlier IS DISTINCT FROM 1
  AND articulated = 1
""")

con.execute("""
CREATE OR REPLACE VIEW panel_r AS
SELECT *
FROM panel_quality
WHERE line_1600 > 0
  AND line_1410 >= 0
  AND line_1510 >= 0
  AND line_2300 IS NOT NULL
  AND line_2330 IS NOT NULL
""")

In [65]:
con.execute("""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT inn) AS companies,
    COUNT(DISTINCT year) AS years
FROM panel_r
""").df()

,observations,companies,years
0,1095535,407872,6


In [66]:
con.execute("""
COPY (
    SELECT
        p.inn, p.year,
        p.line_1410, p.line_1510, p.line_1600,
        p.line_2300, p.line_2330,
        p.line_2110, p.line_1150,
        prev.line_1150 AS line_1150_lag1,
        p.line_2411, p.line_2410,
        p.age, p.okved, p.region_taxcode,
        p.simplified, p.totals_adjustment
    FROM panel_r p
    LEFT JOIN panel_quality prev
        ON p.inn = prev.inn
       AND p.year = prev.year + 1
)
TO 'data/rfsd_r.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")